In [1]:
import pandas as pd
import pulp
import time

DATA_FILE = "../data/cleaned_supply_chain.xlsx"

print("Loading data...")
orders             = pd.read_excel(DATA_FILE, sheet_name="OrderList")
freight_rates      = pd.read_excel(DATA_FILE, sheet_name="FreightRates")
wh_costs           = pd.read_excel(DATA_FILE, sheet_name="WhCosts")
wh_capacities      = pd.read_excel(DATA_FILE, sheet_name="WhCapacities")
products_per_plant = pd.read_excel(DATA_FILE, sheet_name="ProductsPerPlant")
vmi_customers      = pd.read_excel(DATA_FILE, sheet_name="VmiCustomers")
plant_ports        = pd.read_excel(DATA_FILE, sheet_name="PlantPorts")

wh_capacities.columns = wh_capacities.columns.str.strip()

print(f"  Orders:          {len(orders)}")
print(f"  Freight rates:   {len(freight_rates)}")
print(f"  Plants:          {len(wh_costs)}")
print(f"  Products/Plant:  {len(products_per_plant)}")


Loading data...
  Orders:          6438
  Freight rates:   1540
  Plants:          19
  Products/Plant:  2036


In [2]:
print("Building lookup structures...")

# Plant -> cost per unit
warehouse_cost = dict(zip(wh_costs["WH"], wh_costs["Cost/unit"]))

# Plant -> max orders per day
capacity = dict(zip(wh_capacities["Plant ID"], wh_capacities["Daily Capacity"]))

# Plant -> set of product IDs it can make
plant_products = (
    products_per_plant
    .groupby("Plant Code")["Product ID"]
    .apply(set)
    .to_dict()
)

# Plant -> set of ports it can ship from
plant_port_map = (
    plant_ports
    .groupby("Plant Code")["Port"]
    .apply(set)
    .to_dict()
)

# VMI customer -> set of allowed plants
vmi_plant_map = (
    vmi_customers
    .groupby("Customers")["Plant Code"]
    .apply(set)
    .to_dict()
)

# List of all plants
all_plants = list(capacity.keys())

print(f"  Total plant capacity: {sum(capacity.values())} orders/day")
print(f"  Total orders to assign: {len(orders)}")

Building lookup structures...
  Total plant capacity: 5791 orders/day
  Total orders to assign: 6438


In [3]:
# Build freight lookup dictionary for fast access
freight_lookup = {}

for _, row in freight_rates.iterrows():
    key = (row["Carrier"], row["orig_port_cd"], row["dest_port_cd"], row["svc_cd"])
    entry = (row["minm_wgh_qty"], row["max_wgh_qty"], row["minimum cost"], row["rate"])
    freight_lookup.setdefault(key, []).append(entry)


def get_freight_cost(carrier, origin_port, dest_port, service_level, weight):
    """
    Calculate freight cost for a shipment.
    Returns the cost, or None if no matching rate found.
    CRF service level = zero freight cost.
    """
    if service_level == "CRF":
        return 0.0

    key = (carrier, origin_port, dest_port, service_level)
    bands = freight_lookup.get(key, [])

    for min_wt, max_wt, min_cost, rate in bands:
        if min_wt <= weight <= max_wt:
            return max(min_cost, rate * weight)

    return None


print("Freight cost lookup ready.")
print(f"  Unique freight routes: {len(freight_lookup)}")

Freight cost lookup ready.
  Unique freight routes: 44


In [4]:
print("Building feasible (order, plant, carrier) combinations...")

# Get unique carriers from freight rates
all_carriers = freight_rates["Carrier"].unique().tolist()

# Dictionary of (order_index, plant, carrier) -> total cost
feasible = {}
skipped_orders = []

for idx, row in orders.iterrows():
    product_id    = row["Product ID"]
    customer      = row["Customer"]
    service_level = row["Service Level"]
    weight        = row["Weight"]
    quantity      = row["Unit quantity"]
    dest_port     = row["Destination Port"]

    found_any = False

    for plant in all_plants:
        # Check: plant can make this product
        if product_id not in plant_products.get(plant, set()):
            continue

        # Check: VMI restriction
        if customer in vmi_plant_map:
            if plant not in vmi_plant_map[customer]:
                continue

        # Check: plant has port connections
        ports = plant_port_map.get(plant, set())
        if not ports:
            continue

        for origin_port in ports:
            for carrier in all_carriers:
                # Calculate freight cost
                fc = get_freight_cost(
                    carrier, origin_port, dest_port,
                    service_level, weight
                )
                if fc is None:
                    continue

                # Total cost = warehouse cost + freight cost
                wh_cost = warehouse_cost.get(plant, 0) * quantity
                total_cost = wh_cost + fc

                feasible[(idx, plant, carrier)] = total_cost
                found_any = True

    if not found_any:
        skipped_orders.append(idx)

# Orders that have at least one valid assignment
valid_order_indices = [
    i for i in range(len(orders))
    if i not in skipped_orders
]

print(f"  Feasible assignments: {len(feasible)}")
print(f"  Orders with no valid assignment: {len(skipped_orders)}")
print(f"  Orders to optimize: {len(valid_order_indices)}")

Building feasible (order, plant, carrier) combinations...
  Feasible assignments: 17388
  Orders with no valid assignment: 0
  Orders to optimize: 6438


In [10]:
print("Building MILP model...")

model = pulp.LpProblem("SupplyChain_OrderAssignment", pulp.LpMinimize)

# --- Decision Variables ---
# x[i, p, c] = 1 if order i is assigned to plant p via carrier c
x = {}
for (i, p, c) in feasible:
    x[(i, p, c)] = model.add_variable(name=f"x_{i}_{p}_{c}", cat="Binary")

# overflow[p] = how much plant p exceeds its capacity (continuous, >= 0)
overflow = {}
for plant in all_plants:
    overflow[plant] = model.add_variable(name=f"overflow_{plant}", lowBound=0, cat="Continuous")

# Penalty for exceeding capacity
M_PENALTY = 10000

# --- Objective Function ---
# Minimize: total assignment cost + penalty for capacity overflow
model += (
    pulp.lpSum(feasible[k] * x[k] for k in feasible)
    + pulp.lpSum(M_PENALTY * overflow[p] for p in all_plants)
), "Total_Cost_Plus_Penalty"

# Pre-build index maps for fast constraint construction
vars_by_order = {}
vars_by_plant = {}

for key in feasible:
    i, p, c = key
    vars_by_order.setdefault(i, []).append(key)
    vars_by_plant.setdefault(p, []).append(key)

# --- Constraint 1: Each order assigned to exactly one (plant, carrier) ---
for i in valid_order_indices:
    keys = vars_by_order.get(i, [])
    if keys:
        model += (
            pulp.lpSum(x[k] for k in keys) == 1,
            f"Assign_order_{i}"
        )

# --- Constraint 2: Plant capacity (soft constraint with overflow penalty) ---
for plant in all_plants:
    keys = vars_by_plant.get(plant, [])
    if keys:
        model += (
            pulp.lpSum(x[k] for k in keys) <= capacity[plant] + overflow[plant],
            f"Capacity_{plant}"
        )

print(f"  Binary variables: {len(x)}")
print(f"  Overflow variables: {len(overflow)}")
print(f"  Assignment constraints: {len(valid_order_indices)}")
print(f"  Capacity constraints: {len(all_plants)}")

Building MILP model...
  Binary variables: 17388
  Overflow variables: 19
  Assignment constraints: 6438
  Capacity constraints: 19


In [6]:

print(pulp.listSolvers())

['CyLP', 'GLPK_CMD', 'PYGLPK', 'CPLEX_CMD', 'CPLEX_PY', 'GUROBI', 'GUROBI_CMD', 'MOSEK', 'XPRESS', 'XPRESS', 'XPRESS_PY', 'PULP_CBC_CMD', 'COIN_CMD', 'COINMP_DLL', 'CHOCO_CMD', 'MIPCL_CMD', 'SCIP_CMD', 'FSCIP_CMD', 'SCIP_PY', 'HiGHS', 'HiGHS_CMD', 'COPT', 'COPT_DLL', 'COPT_CMD', 'SAS94', 'SASCAS', 'CUOPT']


In [11]:
print("Solving MILP...")
TIME_LIMIT = 300  # 5 minutes max

start_time = time.time()
solver = pulp.getSolver("HiGHS", msg=True, timeLimit=TIME_LIMIT)
result = model.solve(solver)
solve_time = time.time() - start_time

print(f"\nSolver Status: {pulp.LpStatus[result]}")
print(f"Solve Time: {solve_time:.2f} seconds")
print(f"Objective Value: {pulp.value(model.objective):.2f}")

Solving MILP...

Solver Status: Optimal
Solve Time: 0.98 seconds
Objective Value: 50520581.11


In [12]:
# Extract assignments where x = 1
assignments = []

for (i, p, c), var in x.items():
    if var.varValue is not None and var.varValue > 0.5:
        assignments.append({
            "Order_Index": i,
            "Order_ID": orders.loc[i, "Order ID"],
            "Product_ID": orders.loc[i, "Product ID"],
            "Customer": orders.loc[i, "Customer"],
            "Plant": p,
            "Carrier": c,
            "Cost": feasible[(i, p, c)]
        })

results_df = pd.DataFrame(assignments)

print(f"Total assignments: {len(results_df)}")
print(f"\nSample assignments:")
results_df.head(10)

Total assignments: 6438

Sample assignments:


,Order_Index,Order_ID,Product_ID,Customer,Plant,Carrier,Cost
0,0,1.447296e+09,1700106,V55555_53,PLANT16,V444_9,1551.204466
1,1,1.447158e+09,1700106,V55555_53,PLANT16,V444_9,6120.346335
2,2,1.447139e+09,1700106,V55555_53,PLANT16,V444_9,4475.071301
3,3,1.447364e+09,1700106,V55555_53,PLANT16,V444_9,1626.076959
4,4,1.447364e+09,1700106,V55555_53,PLANT16,V444_9,4152.543639
5,5,1.447351e+09,1700106,V55555_53,PLANT16,V444_9,6396.798616
6,6,1.447320e+09,1700106,V55555_53,PLANT16,V444_9,3421.096979
7,7,1.447158e+09,1700106,V55555_53,PLANT16,V444_9,819.757806
8,8,1.447219e+09,1700106,V55555_53,PLANT16,V444_9,2478.471493
9,9,1.447398e+09,1700106,V55555_53,PLANT16,V444_9,4404.038423


In [13]:
total_warehouse_cost = 0.0
total_freight_cost = 0.0

for _, row in results_df.iterrows():
    i = row["Order_Index"]
    p = row["Plant"]
    qty = orders.loc[i, "Unit quantity"]
    wh = warehouse_cost.get(p, 0) * qty
    fr = feasible[(i, p, row["Carrier"])] - wh
    total_warehouse_cost += wh
    total_freight_cost += fr

total_penalty = sum(
    M_PENALTY * overflow[p].varValue
    for p in all_plants
    if overflow[p].varValue is not None and overflow[p].varValue > 0
)

print("Cost Breakdown")
print("=" * 40)
print(f"  Warehouse cost:  {total_warehouse_cost:>12,.2f}")
print(f"  Freight cost:    {total_freight_cost:>12,.2f}")
print(f"  Penalty cost:    {total_penalty:>12,.2f}")
print(f"  ---------------------------------")
print(f"  TOTAL:           {total_warehouse_cost + total_freight_cost + total_penalty:>12,.2f}")

Cost Breakdown
  Warehouse cost:  9,809,288.81
  Freight cost:       71,292.30
  Penalty cost:    40,640,000.00
  ---------------------------------
  TOTAL:           50,520,581.11
